# 03 - Model Inversion (Reconstructing Training Data)

**Model inversion** turns a classifier's confidence scores against it: by
optimizing an input to maximize the score for a target class, the attacker
recovers a *representative example* of what that class looked like in training - a
prototypical fraud record, or a recognizable handwritten digit - using nothing but
the `/predict` API.

We run **MI-Face** (confidence-maximizing inversion) against **two published
Dreadnode environments**, so you deploy nothing:

| Environment (`task_ref`)      | Modality | What we recover              |
|-------------------------------|----------|------------------------------|
| `ml-extraction-fraud-tabular` | tabular  | a representative record/class|
| `ml-extraction-mnist-image`   | image    | a reconstructed digit/class  |

**Why it matters (CIA).** Inversion is a **Confidentiality** attack on the *training
data*: if a model will hand back a recognizable prototype of a class, it has
memorized enough to leak what it was trained on - faces, medical records, PII - to
anyone with query access. Rich confidence scores make it worse, so it also informs
how much output detail you should expose.

**Algorithm:** MI-Face confidence-maximizing model inversion -
[Fredrikson, Jha & Ristenpart, CCS 2015](https://dl.acm.org/doi/10.1145/2810103.2813677).
Swap `confidence_inversion` for `nes_inversion` to run the gradient-free NES
variant with the same interface.

> **New here? Run [`00_prerequisites.ipynb`](../00_prerequisites.ipynb) first** -
> install the CLI (`curl -fsSL https://dreadnode.io/install.sh | bash`), sign in
> (`dn login`), and create a workspace. Everything below streams findings to your
> Dreadnode workspace and draws from your credit balance.

> **Follow along in the docs:** [Model Inversion - the Learning Guide](https://docs.dreadnode.io/ai-red-teaming/learning-guide/model-inversion) covers the concept, the threat model, and the defenses in depth.

## Setup

## What we are assessing

We are going to assess **model inversion**: given only a classifier's confidence
scores, can an attacker reconstruct a representative training input for each class -
recovering what the model "remembers" about its data. We invert the fraud (tabular)
and MNIST (image) classifiers and measure the reconstruction confidence per class,
tabular first, then image.

In [ ]:
# Traditional-ML notebooks train scikit-learn / torch surrogate models.
# If this fails, install the extra:  pip install "dreadnode[airt-ml]"
try:
    import sklearn  # noqa: F401
    import torch  # noqa: F401
except ModuleNotFoundError as exc:
    raise SystemExit(
        f"Missing '{exc.name}'. The traditional-ML notebooks need the airt-ml extra:\n"
        '  pip install "dreadnode[airt-ml]"'
    ) from exc

PROJECT = "airt-learning-03-inversion"
ORG = "your-org-slug"  # your organization slug from the platform URL
WORKSPACE = "main"

In [ ]:
import dreadnode as dn

instance = dn.configure(project=PROJECT, workspace=WORKSPACE, organization=ORG, console=False)
api = instance.api
print("configured; findings stream to project:", PROJECT)

In [ ]:
import httpx

from dreadnode.airt import PredictionTargetSpec, confidence_inversion
from dreadnode.airt.assessment import Assessment

In [ ]:
import asyncio
import httpx

from dreadnode.core.environment import TaskEnvironment


_ENVS: list[TaskEnvironment] = []


async def provision(task_ref: str, timeout: int = 180) -> tuple[TaskEnvironment, str]:
    """Spin up a published Dreadnode environment and return (env, base_url) once
    the classifier service is actually answering. `setup()` returns before the app
    binds its port, so we poll /pool until it responds."""
    env = TaskEnvironment(api, org=ORG, workspace=WORKSPACE, task_ref=task_ref)
    _ENVS.append(env)
    ctx = await env.setup()
    url = ctx["service_urls"]["challenge"]["url"]
    for _ in range(timeout // 5):
        try:
            if httpx.get(f"{url}/pool?n=1", timeout=15).status_code == 200:
                print(f"{task_ref} ready at {url}")
                return env, url
        except httpx.HTTPError:
            pass
        await asyncio.sleep(5)
    raise RuntimeError(f"{task_ref} did not become ready in {timeout}s")


def make_spec(url: str, num_classes: int, modality: str, name: str) -> PredictionTargetSpec:
    """Point a PredictionTargetSpec at the environment's /predict endpoint. Text
    targets take {"text": ...}; tabular/image targets take {"features": [...]}."""
    template = '{"text": "{input}"}' if modality == "text" else '{"features": {input}}'
    return PredictionTargetSpec(
        endpoint=f"{url}/predict",
        request_template=template,
        probabilities_path="$.probabilities",
        input_format="text" if modality == "text" else "json_array",
        num_classes=num_classes,
        name=name,
    )

In [ ]:
from IPython.display import HTML, display


def show_reconstructions(result, title: str) -> None:
    """Show what the attack reconstructed for each class - rendered digits for
    image targets, feature previews for tabular - so an audience sees the leak."""
    print(f"\n=== {title} ===")
    print(
        f"mean_confidence={result.mean_confidence:.3f}   "
        f"classes_reconstructed={result.classes_reconstructed}/{result.num_classes}   "
        f"queries={result.query_count}"
    )
    imaged = [c for c in result.per_class if c.get("reconstruction_image")]
    if imaged:
        html = '<div style="display:flex;gap:16px;flex-wrap:wrap;font-family:sans-serif">'
        for c in imaged:
            html += (
                f'<figure style="margin:0"><figcaption>class {c["class"]} '
                f'(conf {c["achieved_confidence"]:.2f})</figcaption>'
                f'<img src="{c["reconstruction_image"]}" width="96" '
                'style="image-rendering:pixelated"></figure>'
            )
        display(HTML(html + "</div>"))
    else:
        for c in result.per_class:
            print(f"  class {c['class']}: confidence={c['achieved_confidence']:.3f}  {c.get('reconstruction_preview', '')}")

## Tabular - reconstruct a representative record per class

For each class (legitimate, fraud) MI-Face searches feature space for the input
the model is most confident belongs to that class. The result is the model's
internal "prototype" for that class - a privacy leak about its training data.

In [ ]:
env, url = await provision("ml-extraction-fraud-tabular")
spec = make_spec(url, num_classes=2, modality="tabular", name="Credit-card fraud (tabular)")
pool = httpx.get(f"{url}/pool?n=50", timeout=60).json()["inputs"]

async with Assessment("confidence_inversion - fraud - dreadnode-env"):
    result = await confidence_inversion(
        spec, num_classes=2, input_dim=len(pool[0]), modality="tabular",
        target_classes=[0, 1], max_queries=1200, seed=0,
        airt_target_model="Credit-card fraud (tabular)",
    ).run()
show_reconstructions(result, "Tabular - representative record per class")
await env.teardown()

## Image - reconstruct a handwritten digit per class

Same attack, image modality. MI-Face optimizes an 8x8 digit image until the model
is confident it is the target digit. The reconstructions render below - you can
literally see the recovered digit for each class.

In [ ]:
env, url = await provision("ml-extraction-mnist-image")
spec = make_spec(url, num_classes=10, modality="image", name="Handwritten digits (image)")

async with Assessment("confidence_inversion - mnist - dreadnode-env"):
    result = await confidence_inversion(
        spec, num_classes=10, input_shape=(8, 8), modality="image",
        target_classes=[0, 3, 7], max_queries=2500, seed=0,
        airt_target_model="Handwritten digits (image)",
    ).run()
show_reconstructions(result, "Image - reconstructed digit per class")
await env.teardown()

## Read the findings

Open **[https://app.dreadnode.io](https://app.dreadnode.io)** -> your workspace -> project
**airt-learning-03-inversion**. Each finding shows the reconstructed input per
class, the confidence the model assigned it, the query budget spent, and (when
reference samples are available) how closely the reconstruction matches a real
member of that class. High confidence on a recognizable reconstruction means the
model memorized enough to leak what its training data looked like.

## Homework

- **Recover more classes:** raise `max_queries` and widen `target_classes` on MNIST.
  Which digits reconstruct cleanly and which stay noisy - and why might a class be
  harder to invert?
- **Confidence = leakage:** re-run against a target that returns only top-1 labels
  (no probabilities). How much does inversion degrade? That is the privacy value of
  *not* exposing raw confidence scores.
- **NES variant:** swap in `nes_inversion` and compare reconstruction quality and
  query count to MI-Face on the same classes.

## Clean up

Hosted environments keep billing compute until you release them. Tear down everything this notebook provisioned:

In [ ]:
for _e in _ENVS:
    await _e.teardown()
print(f"tore down {len(_ENVS)} environment(s)")

## Run it without a notebook (TUI)

Everything here is also driveable from the terminal - same platform, same findings:

- **TUI:** run `dreadnode` (no arguments) for the interactive terminal UI, pick the
  target and attack, and watch progress live.
